In [ ]:
!apt-get update
!apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,808 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy

In [ ]:
!zstd --version

*** zstd command line interface 64-bits v1.4.8, by Yann Collet ***


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time

# Kill any existing ollama process
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

# Start ollama serve in background
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**subprocess.os.environ, "OLLAMA_HOST": "0.0.0.0:11434"}
)
time.sleep(3)

# Verify Ollama is running
!ollama list
print("✅ Ollama is running!")



NAME    ID    SIZE    MODIFIED 
✅ Ollama is running!


In [ ]:
!ollama pull qwen2.5:7b

# Verify it works
!ollama list


NAME          ID              SIZE      MODIFIED               
qwen2.5:7b    845dbda0ea48    4.7 GB    Less than a second ago    


In [ ]:
!ollama run qwen2.5:7b "Say hello in one sentence" --nowordwrap


Hello there! How can I assist you today?



In [ ]:
!pip install crewai crewai-tools fastapi uvicorn pydantic python-dotenv pyyaml requests pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 7.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.9/185.9 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.5/811.5 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install tavily-python

In [ ]:

from __future__ import annotations

import os
import sys
import time
import subprocess
import traceback
from pathlib import Path
from datetime import datetime, timezone

import uvicorn
import yaml
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process, LLM
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

load_dotenv()

False

In [ ]:
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL_NAME = "ollama/qwen2.5:7b"
PORT = int(os.getenv("PORT", "8000"))
BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR  / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
BASE_DIR = Path.cwd()  # /content in Google Colab

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(OUTPUT_DIR)

/content/outputs


In [ ]:
def setup_ollama() -> bool:
    ollama_installed = subprocess.run(
        ["which", "ollama"], capture_output=True
    ).returncode == 0

    if not ollama_installed:
        print("Installing Ollama...")
        result = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True, capture_output=True, text=True,
        )
        if result.returncode != 0:
            print(f"Install failed: {result.stderr[-500:]}")
            return False
        print("Ollama installed.")
    else:
        print("Ollama already installed.")

    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    time.sleep(2)

    env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"}
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=env,
    )
    time.sleep(4)

    try:
        import requests as req
        resp = req.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        models = [m["name"] for m in resp.json().get("models", [])]
        print(f"Ollama running. Models: {models}")

        if not any("qwen2.5:7b" in m for m in models):
            print("Downloading qwen2.5:7b (~4.7GB)...")
            dl = subprocess.run(
                ["ollama", "pull", "qwen2.5:7b"],
                capture_output=True, text=True,
            )
            if dl.returncode != 0:
                print(f"Download failed: {dl.stderr[-300:]}")
                return False
            print("Model downloaded.")
        else:
            print("qwen2.5:7b already available.")
        return True
    except Exception as e:
        print(f"Cannot reach Ollama at {OLLAMA_HOST}: {e}")
        return False


In [ ]:
def get_agents() -> dict:
    return {
        "market_research_specialist": {
            "role": "Senior Market Research Analyst",
            "goal": (
                "Conduct thorough market research for the given product idea. "
                "Identify market size (TAM, SAM, SOM), key growth trends, "
                "regulatory landscape, and market dynamics. Provide data-driven "
                "insights with specific numbers and credible sources where possible."
            ),
            "backstory": (
                "You are a senior market research analyst with 15 years of experience "
                "at top-tier firms like McKinsey and Gartner. You specialize in AI and tech "
                "markets and have a deep understanding of market sizing methodologies, "
                "competitive landscapes, and regulatory frameworks across regions. "
                "You always back your claims with data points, cite credible sources, and "
                "present findings in a structured format. You are thorough but never pad "
                "your analysis with fluff. When you don't have exact data, you clearly "
                "state estimates and explain your reasoning."
            ),
        },
        "competitive_intelligence_analyst": {
            "role": "Competitive Intelligence Analyst",
            "goal": (
                "Identify and analyze the top competitors in the market for the given "
                "product idea. For each competitor, evaluate their product features, "
                "pricing strategy, market share, strengths, weaknesses, and unique "
                "value propositions. Identify gaps in the competitive landscape that "
                "the new product could exploit."
            ),
            "backstory": (
                "You are a competitive intelligence specialist who has analyzed hundreds "
                "of startups and established companies. You have a talent for finding "
                "hidden competitors, understanding their strategies, and identifying "
                "patterns in how markets evolve. You present competitive analysis in "
                "structured comparison tables with clear ratings. You always think about "
                "the competitive moat and what makes each competitor defensible versus "
                "where they are vulnerable. You look at both direct competitors and "
                "adjacent players who could pivot into the space."
            ),
        },
        "customer_insights_researcher": {
            "role": "Customer Insights and User Research Lead",
            "goal": (
                "Define the target customer segments for the product idea. Create "
                "detailed user personas with demographics, psychographics, pain points, "
                "current solutions they use, willingness to pay, and buying behavior. "
                "Identify the most attractive customer segment for initial go-to-market."
            ),
            "backstory": (
                "You are a customer research expert who has conducted thousands of user "
                "interviews and surveys. You excel at turning qualitative insights into "
                "actionable personas. You understand the difference between what people "
                "say they want and what they actually need. You always consider the "
                "economic buyer versus the end user. You segment markets by both demographics "
                "and behavioral patterns. Your personas include specific details like job "
                "titles, salary ranges, daily workflows, and exact pain points with current "
                "solutions rather than vague generalizations."
            ),
        },
        "product_strategy_advisor": {
            "role": "Product Strategy and Roadmap Advisor",
            "goal": (
                "Based on the market research, competitive analysis, and customer insights, "
                "define a comprehensive product strategy. This includes MVP feature set "
                "prioritized by MoSCoW, product differentiation strategy, pricing model, "
                "go-to-market approach, and a phased development roadmap with clear milestones."
            ),
            "backstory": (
                "You are a seasoned product strategist who has launched products at both "
                "startups and Fortune 500 companies. You think in terms of product-market "
                "fit, unit economics, and viral growth loops. You prioritize ruthlessly "
                "because you know building too many features is the number one startup killer. "
                "You use frameworks like MoSCoW, RICE scoring, and Jobs-to-be-Done. Your "
                "roadmaps are concrete with specific milestones and success metrics rather "
                "than vague aspirational timelines. You always tie features back to customer "
                "pain points identified in the research."
            ),
        },
        "business_analyst": {
            "role": "Business Strategy and Financial Analyst",
            "goal": (
                "Synthesize all previous research into a final business recommendation. "
                "Provide a clear Go or No-Go decision with confidence level. Include revenue "
                "projections for three years, cost structure analysis, pricing strategy, "
                "key risks with mitigation plans, and critical success factors."
            ),
            "backstory": (
                "You are a business analyst and startup advisor who has evaluated "
                "hundreds of business plans. You are brutally honest and do not sugarcoat "
                "bad ideas. You think in terms of unit economics, CAC to LTV ratios, "
                "break-even timelines, and market timing. You have seen startups succeed "
                "and fail and you know the patterns that predict each outcome. Your "
                "recommendations are always clear and actionable. You include specific "
                "financial projections with assumptions clearly stated. You identify "
                "the top three to five risks and provide concrete mitigation strategies."
            ),
        },
    }



In [ ]:
def get_tasks() -> dict:
    return {
        "market_research_task": {
            "description": (
                "Conduct a comprehensive market analysis for the following product idea: "
                "{product_idea}\n\n"
                "Your analysis must cover all of the following:\n\n"
                "1. Market Size: Estimate TAM, SAM, and SOM with specific dollar figures "
                "and growth rates. Show your calculation assumptions.\n\n"
                "2. Market Trends: Identify the top 3-4 trends driving this market. "
                "For each trend, explain why it matters and how it creates opportunity.\n\n"
                "3. Regulatory Landscape: List key regulations, compliance requirements, "
                "and legal considerations. Note any upcoming regulatory changes that could "
                "impact this market.\n\n"
                "4. Market Dynamics: Analyze barriers to entry, supplier power, buyer "
                "power, threat of substitutes, and competitive rivalry using a brief "
                "Porter's Five Forces framework.\n\n"
                "5. Timing: Assess whether now is the right time to enter this market "
                "and why.\n\n"
                "Use web search tools if available. If not, use your training knowledge. "
                "Be specific with numbers and avoid vague statements."
            ),
            "expected_output": (
                "A comprehensive market research report of 500-800 words with clear sections "
                "for market size with TAM, SAM, and SOM figures, top 3-4 trends with "
                "explanations, regulatory landscape, Porter's Five Forces summary, and "
                "timing assessment. Use bullet points and bold headings."
            ),
        },
        "competitive_intelligence_task": {
            "description": (
                "Analyze the competitive landscape for: {product_idea}\n\n"
                "Your analysis must include:\n\n"
                "1. Direct Competitors: Identify 4-6 direct competitors. For each provide "
                "company name, founding year, pricing, key features, target segment, "
                "strength, weakness, and funding or revenue if known.\n\n"
                "2. Indirect Competitors: List 2-3 adjacent players who could pivot.\n\n"
                "3. Competitive Matrix: Create a feature comparison table.\n\n"
                "4. Market Gaps: Identify 2-3 unmet needs or underserved segments.\n\n"
                "5. Competitive Moat Assessment: How defensible is each competitor?\n\n"
                "Use web search if available. Be specific with real company names and prices."
            ),
            "expected_output": (
                "A detailed competitive intelligence report of 500-800 words with a "
                "structured comparison table of 4-6 competitors showing name, pricing, "
                "strengths, and weaknesses, plus indirect competitors list, market gap "
                "analysis, and competitive moat assessment."
            ),
        },
        "customer_insights_task": {
            "description": (
                "Develop detailed customer insights for: {product_idea}\n\n"
                "Your analysis must include:\n\n"
                "1. User Personas: Create 3 detailed personas, each with name, age, "
                "job title, industry, location, current tools they use, their number "
                "one pain point, how much they would pay per month, how they would "
                "discover this product, and a quote that captures their mindset.\n\n"
                "2. Segment Attractiveness: Rank personas by market size, willingness "
                "to pay, acquisition ease, and retention potential.\n\n"
                "3. Buying Decision: Map the buying journey from awareness through "
                "consideration, decision, adoption, to advocacy.\n\n"
                "4. Switching Costs: What would make users switch from their current "
                "solution?\n\n"
                "Use web search if available. Make personas realistic and specific."
            ),
            "expected_output": (
                "A detailed customer insights report of 500-800 words with 3 fully "
                "fleshed-out user personas including demographics, pain points, budget "
                "ranges, and discovery channels, plus segment attractiveness ranking, "
                "buying journey map, and switching cost analysis."
            ),
        },
        "product_strategy_task": {
            "description": (
                "Based on all previous research covering market, competition, and "
                "customers, define the product strategy for: {product_idea}\n\n"
                "Your strategy must include:\n\n"
                "1. Value Proposition: A clear one-sentence value prop and 3 supporting "
                "points that differentiate from competitors.\n\n"
                "2. MVP Feature Set using MoSCoW prioritization:\n"
                "   - 5-7 MUST-HAVE features with brief descriptions\n"
                "   - 3-4 SHOULD-HAVE features\n"
                "   - 2-3 COULD-HAVE features\n"
                "   - 1-2 WON'T-HAVE with explicit reasons\n\n"
                "3. Differentiation Strategy: How will this product stand out and "
                "what is the unique angle?\n\n"
                "4. Pricing Model: Whether free, freemium, subscription, or usage-based "
                "with specific price points and justification.\n\n"
                "5. 3-Phase Roadmap:\n"
                "   - Phase 1 Months 1-3: MVP launch with specific features and milestone\n"
                "   - Phase 2 Months 4-8: Growth with specific features and KPI targets\n"
                "   - Phase 3 Months 9-18: Scale with specific features and revenue targets\n\n"
                "6. Go-to-Market Strategy: Top 3 channels to acquire the first 1000 users.\n\n"
                "Tie every feature back to a specific customer pain point from the research."
            ),
            "expected_output": (
                "A comprehensive product strategy document of 600-1000 words with clear "
                "value proposition, MoSCoW-prioritized feature set with descriptions, "
                "differentiation strategy, pricing model with justification, 3-phase "
                "roadmap with specific milestones and metrics, and go-to-market plan "
                "for the first 1000 users."
            ),
        },
        "business_analyst_task": {
            "description": (
                "Write the final business verdict for: {product_idea}\n\n"
                "Synthesize all previous research into a clear recommendation. "
                "Your analysis must include:\n\n"
                "1. Executive Summary: 3-4 sentences summarizing the opportunity.\n\n"
                "2. Go/No-Go Recommendation: Clear verdict of GO, CONDITIONAL GO, or "
                "NO-GO with confidence level of High, Medium, or Low and a 2-3 "
                "sentence rationale.\n\n"
                "3. Revenue Projections for 3 years:\n"
                "   - Year 1: Users, revenue, key assumptions\n"
                "   - Year 2: Users, revenue, key assumptions\n"
                "   - Year 3: Users, revenue, key assumptions\n\n"
                "4. Cost Structure: Major cost categories including development, "
                "marketing, infrastructure, and operations with estimated monthly burn.\n\n"
                "5. Unit Economics: Estimated CAC, LTV, LTV to CAC ratio, and payback "
                "period.\n\n"
                "6. Top 5 Risks: For each risk provide a description, probability of "
                "High, Medium, or Low, impact of High, Medium, or Low, and a specific "
                "mitigation strategy.\n\n"
                "7. Critical Success Factors: The top 3-4 things that must go right.\n\n"
                "8. Final Recommendation: One paragraph honest assessment. If this is "
                "a bad idea, say so. If it is good but timing is wrong, say that too.\n\n"
                "Be brutally honest. It is better to kill a bad idea early than waste "
                "months on it."
            ),
            "expected_output": (
                "A comprehensive business analysis report of 600-1000 words with "
                "executive summary, clear Go/No-Go recommendation with confidence level, "
                "3-year revenue projections with assumptions, cost structure breakdown, "
                "unit economics covering CAC, LTV, and payback, top 5 risks with "
                "mitigations, critical success factors, and an honest final verdict."
            ),
        },
    }



In [ ]:
def build_llm() -> LLM:
    return LLM(
        model=MODEL_NAME,
        base_url=OLLAMA_HOST,
        temperature=0.7,
        max_tokens=2000,
    )



In [ ]:
def build_tools() -> list:
    from google.colab import userdata
    tavily_key = userdata.get("TAVILY_API_KEY")
    if tavily_key:
        os.environ["TAVILY_API_KEY"] = tavily_key

    if tavily_key:
        try:
            from crewai_tools import TavilySearchTool
            tool = TavilySearchTool(max_results=5, topic="general")
            print("Tavily search tool enabled.")
            return [tool]
        except Exception as e:
            print(f"Tavily init failed: {e}. Running without web search.")
    return []


In [ ]:
def build_crew(product_idea: str, verbose: bool = False) -> Crew:
    agents_cfg = get_agents()
    tasks_cfg = get_tasks()
    llm = build_llm()
    tools = build_tools()

    agents = {}
    for key, cfg in agents_cfg.items():
        agent_kwargs = {
            "role": cfg["role"],
            "goal": cfg["goal"],
            "backstory": cfg["backstory"],
            "llm": llm,
            "verbose": verbose,
            "allow_delegation": False,
        }
        if tools:
            agent_kwargs["tools"] = tools
        agents[key] = Agent(**agent_kwargs)

    market_task = Task(
        description=tasks_cfg["market_research_task"]["description"].format(product_idea=product_idea),
        expected_output=tasks_cfg["market_research_task"]["expected_output"],
        agent=agents["market_research_specialist"],
    )

    competitive_task = Task(
        description=tasks_cfg["competitive_intelligence_task"]["description"].format(product_idea=product_idea),
        expected_output=tasks_cfg["competitive_intelligence_task"]["expected_output"],
        agent=agents["competitive_intelligence_analyst"],
        context=[market_task],
    )

    customer_task = Task(
        description=tasks_cfg["customer_insights_task"]["description"].format(product_idea=product_idea),
        expected_output=tasks_cfg["customer_insights_task"]["expected_output"],
        agent=agents["customer_insights_researcher"],
        context=[market_task, competitive_task],
    )

    strategy_task = Task(
        description=tasks_cfg["product_strategy_task"]["description"].format(product_idea=product_idea),
        expected_output=tasks_cfg["product_strategy_task"]["expected_output"],
        agent=agents["product_strategy_advisor"],
        context=[market_task, competitive_task, customer_task],
    )

    business_task = Task(
        description=tasks_cfg["business_analyst_task"]["description"].format(product_idea=product_idea),
        expected_output=tasks_cfg["business_analyst_task"]["expected_output"],
        agent=agents["business_analyst"],
        context=[market_task, competitive_task, customer_task, strategy_task],
    )

    return Crew(
        agents=list(agents.values()),
        tasks=[market_task, competitive_task, customer_task, strategy_task, business_task],
        process=Process.sequential,
        verbose=True,
        memory=False,
    )


In [ ]:
def run_research(product_idea: str, verbose: bool = False) -> dict:
    start_ts = datetime.now(timezone.utc).isoformat()
    print(f"Building crew for: {product_idea}")
    print(f"LLM: {MODEL_NAME} @ {OLLAMA_HOST}")
    print("Starting pipeline... (~1-3 min per agent)")

    crew = build_crew(product_idea=product_idea, verbose=verbose)
    result = crew.kickoff()
    print("Pipeline finished.")

    task_outputs = []
    for i, t in enumerate(crew.tasks):
        raw = ""
        try:
            if t.output is not None:
                raw = getattr(t.output, "raw", str(t.output)) or ""
        except Exception:
            pass
        task_outputs.append(raw if raw else f"(Agent {i + 1} produced no output)")

    final_report = ""
    try:
        if result is not None:
            final_report = getattr(result, "raw", str(result)) or ""
    except Exception:
        pass
    if not final_report:
        final_report = task_outputs[-1] if task_outputs else "No output produced."

    safe_name = "".join(c if c.isalnum() or c in "-_" else "_" for c in product_idea)[:60]
    run_dir = OUTPUT_DIR / f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{safe_name}"
    run_dir.mkdir(parents=True, exist_ok=True)

    labels = [
        "01_market_research.md",
        "02_competitive_intelligence.md",
        "03_customer_insights.md",
        "04_product_strategy.md",
        "05_business_analysis.md",
        "final_report.md",
    ]
    contents = [*task_outputs, final_report]
    for label, content in zip(labels, contents):
        (run_dir / label).write_text(content, encoding="utf-8")

    print(f"Outputs saved to: {run_dir}")

    return {
        "product_idea": product_idea,
        "final_report": final_report,
        "market_research": task_outputs[0] if len(task_outputs) > 0 else "",
        "competitive_intelligence": task_outputs[1] if len(task_outputs) > 1 else "",
        "customer_insights": task_outputs[2] if len(task_outputs) > 2 else "",
        "product_strategy": task_outputs[3] if len(task_outputs) > 3 else "",
        "business_analysis": task_outputs[4] if len(task_outputs) > 4 else "",
        "generated_at": start_ts,
        "output_path": str(run_dir),
    }


In [ ]:
app = FastAPI(title="Multi-Agent AI Research Assistant", version="3.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class ReportRequest(BaseModel):
    product_idea: str = Field(..., min_length=5, max_length=1000)
    verbose: bool = Field(default=False)


class ReportResponse(BaseModel):
    product_idea: str
    final_report: str
    market_research: str
    competitive_intelligence: str
    customer_insights: str
    product_strategy: str
    business_analysis: str
    generated_at: str
    output_path: str



In [ ]:
@app.get("/")
def health():
    ollama_running = False
    models = []
    try:
        import requests as req
        resp = req.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        models = [m.get("name", "") for m in resp.json().get("models", [])]
        ollama_running = True
    except Exception:
        pass

    tavily_key = os.getenv("TAVILY_API_KEY", "").strip()
    return {
        "status": "ok",
        "llm": {
            "provider": "ollama",
            "model": MODEL_NAME,
            "base_url": OLLAMA_HOST,
            "running": ollama_running,
            "models": models,
        },
        "tavily_enabled": bool(
            tavily_key and tavily_key != "your_tavily_api_key_here"
        ),
    }

In [ ]:
@app.get("/validate")
def validate():
    errors = []
    warnings = []

    try:
        import requests as req
        resp = req.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        models = [m.get("name", "") for m in resp.json().get("models", [])]
        if any("qwen2.5" in m for m in models):
            warnings.append(f"Ollama OK. Models: {models}")
        else:
            errors.append(f"Qwen2.5-7B not found. Available: {models}")
    except Exception as e:
        errors.append(f"Ollama not reachable at {OLLAMA_HOST}: {e}")

    warnings.append(f"Agents: {len(get_agents())} defined")
    warnings.append(f"Tasks: {len(get_tasks())} defined")

    tavily = os.getenv("TAVILY_API_KEY", "").strip()
    if tavily and tavily != "your_tavily_api_key_here":
        warnings.append("TAVILY_API_KEY set, web search enabled")
    else:
        warnings.append("TAVILY_API_KEY not set, web search disabled")

    return {"ok": len(errors) == 0, "errors": errors, "warnings": warnings}

In [ ]:
@app.post("/generate-report", response_model=ReportResponse)
def generate_report(req: ReportRequest):
    try:
        result = run_research(req.product_idea, verbose=req.verbose)
        return ReportResponse(**result)
    except FileNotFoundError as e:
        raise HTTPException(status_code=404, detail=str(e))
    except KeyError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Pipeline error: {e}")



In [ ]:
def main_cli():
    if len(sys.argv) < 2:
        print('Usage: python main.py "<product idea>" [--verbose]')
        sys.exit(1)
    product_idea = sys.argv[1]
    verbose = "--verbose" in sys.argv or "-v" in sys.argv
    result = run_research(product_idea, verbose=verbose)
    print("\n" + "=" * 80)
    print("FINAL REPORT")
    print("=" * 80)
    print(result["final_report"])
    print(f"\nArtifacts saved to: {result['output_path']}")
    print("=" * 80)


In [ ]:
def start_ngrok():
    try:
        from pyngrok import ngrok
    except ImportError:
        print("pyngrok not installed. Run: pip install pyngrok")
        return None

    token = "39TguaIsciL3QgLxizCiv8k0Vz4_59fESS52UMS6bcKieEon6"
    if not token:
        print("NGROK_TOKEN not set. Server runs locally only.")
        return None

    ngrok.set_auth_token(token)
    try:
        ngrok.kill()
    except Exception:
        pass
    time.sleep(1)

    url = ngrok.connect(addr=PORT, bind_tls=True)
    return str(url).replace("tcp://", "https://")



In [ ]:
!pip install nest_asyncio

In [ ]:
# Check if Ollama is already running
ollama_ready = False
try:
    import requests as req
    req.get(f"{OLLAMA_HOST}/api/tags", timeout=3)
    ollama_ready = True
    print("Ollama already running.")
except Exception:
    print("Ollama not detected, setting up...")
    ollama_ready = setup_ollama()

if not ollama_ready:
    print("Ollama setup failed.")
    sys.exit(1)

# Try ngrok
public_url = start_ngrok()
if public_url:
    print(f"\nNgrok URL: {public_url}")
    print("Paste this URL into your Streamlit app.\n")

print(f"Server:   http://localhost:{PORT}")
print(f"Docs:     http://localhost:{PORT}/docs")
print(f"Validate: http://localhost:{PORT}/validate")
if public_url:
    print(f"Public:   {public_url}")

import asyncio
config = uvicorn.Config(app, host="0.0.0.0", port=PORT)
server = uvicorn.Server(config)
await server.serve()

Ollama not detected, setting up...
Ollama already installed.
Ollama running. Models: ['qwen2.5:7b']
qwen2.5:7b already available.

Ngrok URL: NgrokTunnel: "https://transgressive-archimedes-vixenly.ngrok-free.dev" -> "http://localhost:8000"
Paste this URL into your Streamlit app.

Server:   http://localhost:8000
Docs:     http://localhost:8000/docs
Validate: http://localhost:8000/validate
Public:   NgrokTunnel: "https://transgressive-archimedes-vixenly.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [5368]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Building crew for: Ai Resume Builder
LLM: ollama/qwen2.5:7b @ http://localhost:11434
Starting pipeline... (~1-3 min per agent)
Tavily search tool enabled.


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5453f43e-2253-4d2f-8b89-00d7668d0ac5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Conduct a comprehensive market analysis for the following product idea: Ai Resume Builder                │
│                                                                                                                 │
│  Your analysis must cover all of the following:                                                                 │
│                                                                                                                 │
│  1. Market Size: Estimate TAM, SAM, and SOM with specific dollar figures and growth rates. Show your            │
│  calculation assumptions.                                                                                       │
│                                                                                                                 │
│  2. Market Trends: Identify the top 3-4 trends driving this market. For each trend, explain why it matters and  │
│  how it creates opportunity.                                                                                    │
│                                                                                                                 │
│  3. Regulatory Landscape: List key regulations, compliance requirements, and legal considerations. Note any     │
│  upcoming regulatory changes that could impact this market.                                                     │
│                                                                                                                 │
│  4. Market Dynamics: Analyze barriers to entry, supplier power, buyer power, threat of substitutes, and         │
│  competitive rivalry using a brief Porter's Five Forces framework.                                              │
│                                                                                                                 │
│  5. Timing: Assess whether now is the right time to enter this market and why.                                  │
│                                                                                                                 │
│  Use web search tools if available. If not, use your training knowledge. Be specific with numbers and avoid     │
│  vague statements.                                                                                              │
│  ID: c41ce963-31ab-4dd6-bb20-00390a798fed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Strategy and Financial Analyst                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Comprehensive Product Strategy Document                                                                    │
│                                                                                                                 │
│  #### 1. Value Proposition                                                                                      │
│                                                                                                                 │
│  **AI Resume Builder: Craft Your Perfect Resume with AI-Powered Insights**                                      │
│  - **Supporting Points:**                                                                                       │
│    - **Personalized Recommendations**: Tailored resume content based on job descriptions.                       │
│    - **Time-Saving Automation**: Quickly optimize your resume with minimal effort.                              │
│    - **ATS Compatibility**: Ensure your resume stands out in applicant tracking systems.                        │
│                                                                                                                 │
│  #### 2. MVP Feature Set (MoSCoW Method)                                                                        │
│                                                                                                                 │
│  - **MUST-HAVE Features:**                                                                                      │
│    1. **Job Description Analysis Tool**: Automatically extract relevant keywords and phrases from job           │
│  postings.                                                                                                      │
│    2. **Resume Editor with AI Recommendations**: Real-time suggestions for improvement based on analysis of     │
│  your resume.                                                                                                   │
│    3. **ATS Compatibility Checker**: Ensure your resume is optimized for ATS systems.                           │
│                                                                                                                 │
│  - **SHOULD-HAVE Features:**                                                                                    │
│    4. **Customizable Templates**: A variety of professionally designed templates aligned with different         │
│  industries and job roles.                                                                                      │
│    5. **Automated Job Matching Engine**: Suggest suitable job opportunities based on your skills and            │
│  experience.                                                                                                    │
│                                                                                                                 │
│  - **COULD-HAVE Features (Optional):**                                                                          │
│    6. **Language Support**: Multi-language support for international users.                                     │
│    7. **Analytics Dashboard**: Detailed insights into application performance, keyword usage, and more.         │
│                                                                                                                 │
│  - **WONT-HAVE Features:**                             

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write the final business verdict for: Ai Resume Builder                                                  │
│                                                                                                                 │
│  Synthesize all previous research into a clear recommendation. Your analysis must include:                      │
│                                                                                                                 │
│  1. Executive Summary: 3-4 sentences summarizing the opportunity.                                               │
│                                                                                                                 │
│  2. Go/No-Go Recommendation: Clear verdict of GO, CONDITIONAL GO, or NO-GO with confidence level of High,       │
│  Medium, or Low and a 2-3 sentence rationale.                                                                   │
│                                                                                                                 │
│  3. Revenue Projections for 3 years:                                                                            │
│     - Year 1: Users, revenue, key assumptions                                                                   │
│     - Year 2: Users, revenue, key assumptions                                                                   │
│     - Year 3: Users, revenue, key assumptions                                                                   │
│                                                                                                                 │
│  4. Cost Structure: Major cost categories including development, marketing, infrastructure, and operations      │
│  with estimated monthly burn.                                                                                   │
│                                                                                                                 │
│  5. Unit Economics: Estimated CAC, LTV, LTV to CAC ratio, and payback period.                                   │
│                                                                                                                 │
│  6. Top 5 Risks: For each risk provide a description, probability of High, Medium, or Low, impact of High,      │
│  Medium, or Low, and a specific mitigation strategy.                                                            │
│                                                                                                                 │
│  7. Critical Success Factors: The top 3-4 things that must go right.                                            │
│                                                                                                                 │
│  8. Final Recommendation: One paragraph honest assessment. If this is a bad idea, say so. If it is good but     │
│  timing is wrong, say that too.                                                                                 │
│                                                                                                                 │
│  Be brutally honest. It is better to kill a bad idea early than waste months on it.                             │
│  Agent: Business Strategy and Financial Analyst                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

Pipeline finished.
Outputs saved to: /content/outputs/run_20260708_195222_Ai_Resume_Builder
INFO:     103.75.246.232:0 - "POST /generate-report HTTP/1.1" 200 OK


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5453f43e-2253-4d2f-8b89-00d7668d0ac5                                                                       │
│  Final Output: ### Comprehensive Product Strategy Document                                                      │
│                                                                                                                 │
│  #### 1. Value Proposition                                                                                      │
│                                                                                                                 │
│  **AI Resume Builder: Craft Your Perfect Resume with AI-Powered Insights**                                      │
│  - **Supporting Points:**                                                                                       │
│    - **Personalized Recommendations**: Tailored resume content based on job descriptions.                       │
│    - **Time-Saving Automation**: Quickly optimize your resume with minimal effort.                              │
│    - **ATS Compatibility**: Ensure your resume stands out in applicant tracking systems.                        │
│                                                                                                                 │
│  #### 2. MVP Feature Set (MoSCoW Method)                                                                        │
│                                                                                                                 │
│  - **MUST-HAVE Features:**                                                                                      │
│    1. **Job Description Analysis Tool**: Automatically extract relevant keywords and phrases from job           │
│  postings.                                                                                                      │
│    2. **Resume Editor with AI Recommendations**: Real-time suggestions for improvement based on analysis of     │
│  your resume.                                                                                                   │
│    3. **ATS Compatibility Checker**: Ensure your resume is optimized for ATS systems.                           │
│                                                                                                                 │
│  - **SHOULD-HAVE Features:**                                                                                    │
│    4. **Customizable Templates**: A variety of professionally designed templates aligned with different         │
│  industries and job roles.                                                                                      │
│    5. **Automated Job Matching Engine**: Suggest suitable job opportunities based on your skills and            │
│  experience.                                                                                                    │
│                                                                                                                 │
│  - **COULD-HAVE Features (Optional):**                                                                          │
│    6. **Language Support**: Multi-language support for international users.                                     │
│    7. **Analytics Dashboard**: Detailed insights into application performance, keyword usage, and more.         │
│                                                                                                                 │
│  - **WONT-HAVE Features:**                            

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯